# Decision Tree Regression — White Wine Quality

This notebook practices a basic regression workflow using the white wine quality dataset. The target is `quality`; the input features are the other 11 columns.

**Dataset requirement:** place `whitewines.csv` in the same folder as this notebook before running it.


## 1. Import libraries and load the data


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error


In [ ]:
df = pd.read_csv("whitewines.csv")

print("Dataset shape:", df.shape)
df.head()


## 2. Inspect and clean the data

Check the data types, missing values, summary statistics, and exact duplicate rows before modelling.


In [ ]:
df.info()


In [ ]:
print("Missing values by column:")
print(df.isna().sum())


In [ ]:
df.describe()


In [ ]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Cleaned dataset shape:", df.shape)


## 3. Separate features and target

- `X` contains every input feature except `quality`.
- `y` contains the quality values we want to predict.


In [ ]:
X = df.drop(columns=["quality"])
y = df["quality"]

print("X:", X.shape)
print("y:", y.shape)


## 4. Create training and test sets

Keep the test set separate for the final evaluation. We'll use a validation split from the training data to compare tree depths.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


## 5. Create fitting and validation sets

The validation set helps us compare model settings. The test set remains separate until the final evaluation.


In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print("X_fit:", X_fit.shape)
print("X_val:", X_val.shape)
print("y_fit:", y_fit.shape)
print("y_val:", y_val.shape)


## 6. Train an unrestricted Decision Tree

`DecisionTreeRegressor()` with the default `max_depth=None` can grow a very deep tree. Measure both fitting and validation MAE to see how it generalizes.


In [ ]:
baseline_tree = DecisionTreeRegressor(random_state=42)
baseline_tree.fit(X_fit, y_fit)

fit_predictions = baseline_tree.predict(X_fit)
val_predictions = baseline_tree.predict(X_val)

baseline_train_mae = mean_absolute_error(y_fit, fit_predictions)
baseline_val_mae = mean_absolute_error(y_val, val_predictions)

print("Tree depth:", baseline_tree.get_depth())
print("Number of leaves:", baseline_tree.get_n_leaves())
print("Training MAE:", baseline_train_mae)
print("Validation MAE:", baseline_val_mae)


## 7. Compare `max_depth=3` and `max_depth=2`

`max_depth` limits how many split levels the tree can grow. Compare both settings on the **same validation set**. Lower MAE is better.


In [ ]:
tree_depth_3 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_depth_3.fit(X_fit, y_fit)

train_pred_3 = tree_depth_3.predict(X_fit)
val_pred_3 = tree_depth_3.predict(X_val)
train_mae_3 = mean_absolute_error(y_fit, train_pred_3)
val_mae_3 = mean_absolute_error(y_val, val_pred_3)

print("Depth 3 — training MAE:", train_mae_3)
print("Depth 3 — validation MAE:", val_mae_3)
print("Depth 3 — actual depth:", tree_depth_3.get_depth())
print("Depth 3 — number of leaves:", tree_depth_3.get_n_leaves())


In [ ]:
tree_depth_2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_depth_2.fit(X_fit, y_fit)

train_pred_2 = tree_depth_2.predict(X_fit)
val_pred_2 = tree_depth_2.predict(X_val)
train_mae_2 = mean_absolute_error(y_fit, train_pred_2)
val_mae_2 = mean_absolute_error(y_val, val_pred_2)

print("Depth 2 — training MAE:", train_mae_2)
print("Depth 2 — validation MAE:", val_mae_2)
print("Depth 2 — actual depth:", tree_depth_2.get_depth())
print("Depth 2 — number of leaves:", tree_depth_2.get_n_leaves())


In [ ]:
comparison = pd.DataFrame({
    "max_depth": [2, 3],
    "training_MAE": [train_mae_2, train_mae_3],
    "validation_MAE": [val_mae_2, val_mae_3],
})
comparison


**Decision from this experiment:** depth 3 had lower validation MAE than depth 2, so we select `max_depth=3` from these two candidates. This does not prove depth 3 is the best possible setting; it is the better of the settings compared here.


## 8. Train the selected model and evaluate it on the test set

Now train a fresh model using the full training split. Use the test data only for this final evaluation.


In [ ]:
final_model = DecisionTreeRegressor(max_depth=3, random_state=42)
final_model.fit(X_train, y_train)

test_predictions = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_predictions)

print("Final test MAE:", test_mae)


## 9. What we learned

- `X` contains the input features; `y` contains the target.
- `.fit()` trains the model, `.predict()` generates predictions, and `mean_absolute_error()` measures average absolute error.
- We used validation MAE to compare `max_depth=2` and `max_depth=3`; depth 3 performed better among those two candidates.
- A very deep tree can fit the training data perfectly and still make larger errors on validation data, which is a warning sign of overfitting.
- Test MAE is the final score for the selected configuration. Lower MAE is better when comparing models on the same evaluation set.
